# 4장 메모리

이전 대화를 저장하고 다시 사용하는 방법을 살펴봅니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 4장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

---

## 1 - 언어 모델 선택

이번 실습에 사용할 언어 모델을 지정합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 대화 메모리 추가

앞 장에서 선택한 모델에 이전 대화를 전달하는 방법을 살펴봅니다.

![대화 기록을 모델에 전달하는 메모리 모듈](../images/ch4.png)

2장에서 정의한 `TinyAgent`는 질문 사이에 대화 상태를 유지하지 않습니다. 먼저 에이전트에게 이름을 알려 줍니다.

In [ ]:
from rich import print
from illustrated_agents.chapters.ch2 import TinyAgent

ch_2_agent = TinyAgent(llm=llm)
response = ch_2_agent.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

이름을 다시 물으면 모델이 기억하지 못한 것처럼 보입니다. 두 번째 호출에서 모델이 받은 입력에는 이름을 알려 준 첫 번째 메시지가 없습니다. 매 호출에 필요한 대화 기록을 함께 전달해야 합니다.

In [ ]:
response = ch_2_agent.run("Hi! What are our names?")
print(response)

실행 기록에는 처음에 알려 준 이름이 남아 있습니다. 하지만 에이전트는 이 기록을 모델 입력에 포함하지 않습니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(ch_2_agent.trajectory)

## 3. `Memory` 모듈

이 예제에서는 대화 이력을 저장하는 메모리를 구현합니다. `Memory`는 다음 호출에 전달할 메시지를 보관합니다. `Trajectory`는 개발자가 동작을 확인하도록 각 단계에서 일어난 일을 더 자세히 기록합니다.

`Memory`는 `messages` 형식으로 대화를 저장합니다. 아래 입력과 응답에서 `role`은 발화자, `content`는 메시지 내용입니다.

```json
[
    {
        "role": "user",
        "content": "Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'."
    },
    {
        "role": "assistant",
        "content": "Hi Maarten and Jay! It's wonderful to meet you."
    }
]
```

사용자가 질문하거나 모델이 응답할 때마다 메시지를 목록 끝에 추가합니다.

In [ ]:
class Memory:
    """대화 이력을 저장하는 기본 메모리이다."""

    def __init__(self):
        self.messages = []

    def add(
        self, role: str, content: str, tool_call: dict | None = None, **kwargs
    ) -> None:
        """메모리에 메시지를 추가한다."""
        message = {"role": role, "content": content}

        # 도구 호출
        if tool_call:
            message["tool_calls"] = [tool_call]

        # 메모리에 메시지 추가
        self.messages.append(message)

    def get_messages(self) -> list[dict]:
        """저장된 메시지 전체를 반환한다."""
        return self.messages

주석이 달린 코드에서 각 메서드가 메시지를 어떻게 다루는지 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import memory_annotated; memory_annotated

## 4. `agent.py` 수정

`TinyAgent`에 메모리를 연결하고 한 단계의 처리를 수정합니다.

1. 사용자의 질문을 `Memory`에 추가합니다.
2. 저장된 메시지를 언어 모델에 보내 답변을 생성합니다.
3. 모델의 답변을 다시 `Memory`에 추가합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import Trajectory

class TinyAgent:
    """교육용 에이전트의 기본 클래스를 정의한다."""

    def __init__(self, llm: LLM, memory: Memory):
        self.llm = llm
        self.memory = memory
        self.tools = None  # 5장에서 도구 추가
        self.planner = None  # 6장에서 계획 기능 추가

        self.trajectory = Trajectory()

    def run(self, task: str) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        self.memory.add("user", task)
        self.trajectory.initialize(task)

        return self._step()

    def _step(self) -> str:
        """에이전트의 한 단계를 실행한다."""
        # 응답을 생성해 메모리에 추가
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response.content)
        self.trajectory.add(response)
        return response.content

    def _execute_action(self, action: str) -> str | None:
        """도구가 수행할 동작을 실행한다."""
        # 이후 장에서 구현할 부분
        return f"Executed action: {action}"

다음 셀에서 세 단계에 해당하는 코드 줄을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import tinyagent_annotated; tinyagent_annotated

다음 비교에서 빨간색은 삭제한 코드이고 초록색은 추가한 코드입니다.

In [ ]:
from illustrated_agents.chapters.ch4 import tinyagents_diff; tinyagents_diff

이제 `Memory`를 사용하도록 `TinyAgent`를 생성합니다.

In [ ]:
# 에이전트에 메모리 추가
memory = Memory()
agent_with_memory = TinyAgent(llm=llm, memory=memory)

앞에서처럼 모델에 말을 걸어 대화 이력에 이름을 남깁니다.

In [ ]:
response = agent_with_memory.run("Hi! We are Maarten and Jay, authors of 'An Illustrated Guide to AI Agents'.")
print(response)

이어서 이름을 물어보고 첫 질문에서 알려 준 이름으로 답하는지 확인합니다.

In [ ]:
response = agent_with_memory.run("Hi! What are our names?")
print(response)

모델이 이름을 답할 수 있었던 이유를 확인하려면 현재 메모리에 저장된 메시지를 살펴봅니다.

In [ ]:
agent_with_memory.memory.get_messages()

새 질문이 들어올 때마다 목록 전체를 모델에 보냅니다. 모델이 앞선 대화를 참조할 수 있는 이유는 프로그램이 그 내용을 입력에 다시 포함하기 때문입니다. `Memory`를 보면 어떤 메시지가 모델에 전달되는지 알 수 있고, `Trajectory`를 보면 각 단계에서 모델이 한 일을 알 수 있습니다.

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent_with_memory.trajectory)

---

# 구현한 내용

대화 기록을 관리하는 `Memory`를 `TinyAgent`에 연결했습니다. 아래에서 언어 모델, 메모리, 에이전트의 구성을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import what_we_built; what_we_built

# 다음 실습

[5장 도구](../chapter05/chapter05.ipynb)로 넘어가거나, 아래 순서로 메모리 실습을 이어 갈 수 있습니다.

1. [단기 메모리](chapter04_short_term_memory.ipynb)에서는 오래된 메시지를 자르거나 대화를 요약합니다.
2. [장기 메모리](chapter04_long_term_memory.ipynb)에서는 검색 증강 생성(RAG)으로 관련 문서를 찾아 모델에 전달합니다.

5장에서는 언어 모델이 도구 호출을 요청하고 프로그램이 그 요청을 실행하는 방법을 배웁니다.